This iPythonNotebook can be used to experiment with the Quantization Aware Training, where the user will be able to introduce quantization to any network. Further, the user can use the quantized model for evaluations on the pytorch framework itself. 

In [ ]:
!pip install netron

import torch
import torch.nn as nn
import edgeai_torchmodelopt
import copy
import netron
import torchvision
from tqdm import tqdm

We define the model, loss function, optimizer and the example input of what the network expects. 

In [ ]:
model = torchvision.models.resnet50(weights='DEFAULT')
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

example_input = torch.rand((1, 3, 224, 224))

y = model(example_input)
print("Output Shape is : {}".format(y.shape))

In [ ]:
model_export_name = "./orig_simple_network_qat.onnx"
torch.onnx.export(model, example_input, model_export_name)
netron.start(model_export_name, 8082)

Here we will be wrapping our model in the QATFxModule which will be responsible for the quantization-aware-training of the models and conversion to the final quantized network. It expects us to pass the number of epochs for which the model need to be trained. It also enables bias calibration of the layers having a bias value, we can set a bias calibration factor (generally 0.01 works well) to enable it. It is suggested to perform QAT for 25-50 epochs, which helps the network stabilise. The epochs counter for the approach gets updated everytime model.train() is called. The general guidelines could be accessed [from here](../edgeai_torchmodelopt/xmodelopt/quantization/v2/docs/guidelines.md).  

In [ ]:
num_epochs = 3
model = edgeai_torchmodelopt.xmodelopt.quantization.v2.QATFxModule(model, backend='qnnpack', bias_calibration_factor=0.01, total_epochs=num_epochs)

Here is the Training Step for the network, where random data is used currently just for an example. **The data, loss and optimizer should be changed to your own dataset.**

In [ ]:
num_train_images = 10
for epoch in range(num_epochs):
    model.train()
    for i in tqdm(range(num_train_images)):
        optimizer.zero_grad()
        output = model(torch.rand(1,3,224,224))
        label = torch.rand(1,1000)
        loss = loss_fn(output, label) 
        loss.backward()
        optimizer.step()

We have the quantized and calibrated 8-bit network now.

In [ ]:
model.eval()
print(model)

In [ ]:
model_export_name = "./converted_simple_network_qat.onnx"
model.export(example_input, model_export_name)
netron.start(model_export_name, 8082)


The netron might show the quantized fused operators as separate because the fake-quantized (Q-DQ) models are exported. 